Before running this file, `processed_data` should already be fully up to date.

This notebook does the following:

1. loads a processed spectrum and the background spectrum
2. converts both to rates
3. subtracts the background rate from the sample rate
4. saves a documentation plot with raw, background, and background-subtracted spectra
5. opens a separate interactive plot using only the background-subtracted spectrum
6. lets the user define fit groups by clicking:
   - one left boundary
   - one or more peak centers
   - one right boundary
   - then pressing `n` to store that group
7. fits each stored group with:
   - one Gaussian + linear background, or
   - multiple Gaussians + one shared linear background
8. lets the user assign theoretical gamma energies (keV) to fitted peaks
9. saves the fitted peak centers, widths, amplitudes, fit windows, and assigned energies to csv

Interactive controls in the fit-selection window:

- left click: add a point
- `n`: finalize current group
- `u`: undo last click in current group
- `d`: delete last completed group
- `r`: reset current unfinished group
- `q`: finish selection and close window
- `h`: print help

In [1]:
# -----------------------------
# FORCE GUI BACKEND
# -----------------------------

import matplotlib

_backend_errors = []

for candidate in ["QtAgg", "TkAgg"]:
    try:
        matplotlib.use(candidate, force=True)
        print(f"Using backend: {candidate}")
        break
    except Exception as e:
        _backend_errors.append((candidate, repr(e)))
else:
    raise RuntimeError(
        "Could not start a GUI matplotlib backend.\n"
        f"Tried: {_backend_errors}\n"
        "Install one of the GUI toolkits, e.g.\n"
        "  conda install pyqt\n"
        "or\n"
        "  conda install tk"
    )

Using backend: QtAgg


In [2]:
# -----------------------------
# IMPORTS
# -----------------------------

import re
import sys
import shutil
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.optimize import curve_fit

plt.rcParams["figure.figsize"] = (10, 6)

print("Python:", sys.version)
print("matplotlib:", matplotlib.__version__)
print("backend:", matplotlib.get_backend())

Python: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:16:10) [GCC 11.2.0]
matplotlib: 3.10.6
backend: QtAgg


In [3]:
# -----------------------------
# FILE LOCATIONS
# -----------------------------

ROOT = Path("processed_data")
BACKGROUND_FILE = ROOT / "calibration" / "background_550PM_feb26_213PM_mar3.csv"

OUTPUT_ROOT = Path("peak_analysis_outputs")
DOC_PLOT_DIR = OUTPUT_ROOT / "documentation_plots"
FIT_PLOT_DIR = OUTPUT_ROOT / "fit_plots"
PER_RUN_DIR = OUTPUT_ROOT / "per_run_csv"
MASTER_FILE = OUTPUT_ROOT / "manual_peak_fits_all.csv"

irrelevant = [
    "_single_test_converted.csv",
    "calibration/TESTFILE_FOR_CONVERSION_BGND.csv",
    "calibration/background_550PM_feb26_213PM_mar3.csv",
    "cs gain setting/Cs_possible_calcurve_feb_24.csv",
    "cs gain setting/Cs_trial_feb_19.csv",
    "calibration_curves/daily_calibration_params.csv",
]

irrelevant_set = set(irrelevant)

def normalize_relpath(path, root=ROOT):
    path = Path(path)
    try:
        rel = path.relative_to(root)
    except ValueError:
        rel = path
    return rel.as_posix()

print("ROOT:", ROOT)
print("BACKGROUND_FILE:", BACKGROUND_FILE)
print("Background exists:", BACKGROUND_FILE.exists())
print("OUTPUT_ROOT:", OUTPUT_ROOT)

ROOT: processed_data
BACKGROUND_FILE: processed_data/calibration/background_550PM_feb26_213PM_mar3.csv
Background exists: True
OUTPUT_ROOT: peak_analysis_outputs


In [4]:
# -----------------------------
# LOAD SPECTRUM / METADATA
# -----------------------------

def _extract_first_number(text):
    if text is None:
        return None

    match = re.search(r"[-+]?\d*\.?\d+", str(text))
    if match is None:
        return None

    try:
        return float(match.group())
    except ValueError:
        return None


def load_spectrum(csv_path):
    """
    Load a processed spectrum csv with preserved metadata and channel data.

    Expected processed structure:
    - metadata lines near the top
    - a line containing 'Channel Data:'
    - then a data table where:
        column 0 = channel / adc
        column 2 = counts
    """

    csv_path = Path(csv_path)

    with open(csv_path, "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

    metadata = {}
    data_start_idx = None

    for i, line in enumerate(lines):
        stripped = line.strip()

        if stripped == "Channel Data:":
            data_start_idx = i + 1
            break

        if stripped.startswith("Elapsed Real Time:"):
            parts = stripped.split(",", 1)
            metadata["Elapsed Real Time"] = parts[1].strip() if len(parts) > 1 else ""
            continue

        if "Elapsed Real Time:" in stripped:
            value = stripped.split("Elapsed Real Time:", 1)[1]
            value = value.split(",", 1)[1].strip() if "," in value else value.strip()
            metadata["Elapsed Real Time"] = value
            continue

        if stripped.startswith("Start Time:"):
            metadata["Start Time"] = stripped.split("Start Time:", 1)[1].lstrip(",").strip()
            continue

        if stripped.startswith("End Time:"):
            metadata["End Time"] = stripped.split("End Time:", 1)[1].lstrip(",").strip()
            continue

        if "," in stripped:
            left, right = stripped.split(",", 1)
            left = left.strip().rstrip(":")
            right = right.strip()
            if left and right and left not in metadata:
                metadata[left] = right

    if data_start_idx is None:
        raise RuntimeError(f"Could not locate 'Channel Data:' block in {csv_path}")

    if data_start_idx < len(lines):
        header_line = lines[data_start_idx].strip().lower()
        if header_line.startswith("channel"):
            data_start_idx += 1

    numeric_rows = []

    for line in lines[data_start_idx:]:
        stripped = line.strip()

        if not stripped:
            continue

        parts = [p.strip() for p in stripped.split(",")]

        if len(parts) < 3:
            continue

        try:
            channel = float(parts[0])
            counts = float(parts[2])
            numeric_rows.append((channel, counts))
        except ValueError:
            continue

    if len(numeric_rows) == 0:
        raise RuntimeError(f"No numeric channel/count data found in {csv_path}")

    arr = np.array(numeric_rows, dtype=float)
    x = arr[:, 0]
    counts = arr[:, 1]

    elapsed_real_time = _extract_first_number(metadata.get("Elapsed Real Time"))
    metadata["Elapsed Real Time (s)"] = elapsed_real_time

    return x, counts, metadata


def counts_to_rate(counts, elapsed_time_seconds):
    if elapsed_time_seconds is None or elapsed_time_seconds <= 0:
        raise RuntimeError("Elapsed real time is missing or nonpositive.")
    return counts / elapsed_time_seconds


def compute_background_subtracted_rate(sample_path, background_path=BACKGROUND_FILE):
    """
    Background subtraction happens BEFORE any peak fitting.
    """

    x_sample, counts_sample, meta_sample = load_spectrum(sample_path)
    x_bg, counts_bg, meta_bg = load_spectrum(background_path)

    if len(x_sample) != len(x_bg) or not np.allclose(x_sample, x_bg):
        raise RuntimeError(
            "Sample and background channel grids do not match. "
            "Background subtraction cannot be performed safely."
        )

    t_sample = meta_sample.get("Elapsed Real Time (s)")
    t_bg = meta_bg.get("Elapsed Real Time (s)")

    raw_rate = counts_to_rate(counts_sample, t_sample)
    background_rate = counts_to_rate(counts_bg, t_bg)
    adjusted_rate = raw_rate - background_rate

    return x_sample, raw_rate, background_rate, adjusted_rate, meta_sample, meta_bg

In [5]:
# -----------------------------
# FIT MODELS
# -----------------------------

def gaussian(x, A, mu, sigma):
    sigma = abs(sigma)
    return A * np.exp(-((x - mu) ** 2) / (2 * sigma ** 2))


def gaussian_with_background(x, A, mu, sigma, m, b):
    return gaussian(x, A, mu, sigma) + m * x + b


def multi_gaussian_with_background(x, *params):
    """
    params for n peaks:
        A1, mu1, sigma1, A2, mu2, sigma2, ..., m, b
    """
    if len(params) < 5:
        raise ValueError("Need at least one Gaussian plus linear background.")

    n_gauss = (len(params) - 2) // 3
    m = params[-2]
    b = params[-1]

    y = m * x + b

    for i in range(n_gauss):
        A = params[3 * i + 0]
        mu = params[3 * i + 1]
        sigma = params[3 * i + 2]
        y = y + gaussian(x, A, mu, sigma)

    return y

In [6]:
# -----------------------------
# FILE LISTING
# -----------------------------

def get_candidate_files(root=ROOT):
    csv_files = sorted(root.rglob("*.csv"))
    kept = []

    for file in csv_files:
        rel = normalize_relpath(file, root)

        if ".ipynb_checkpoints" in rel:
            continue

        if rel in irrelevant_set:
            continue

        kept.append(file)

    return kept


candidate_files = get_candidate_files()

print("Number of candidate files:", len(candidate_files))
for i, file in enumerate(candidate_files):
    print(f"{i:2d} : {normalize_relpath(file)}")

Number of candidate files: 38
 0 : calibration/daily/Co60_21.9C_preamp100_mar5.csv
 1 : calibration/daily/Co60_22.4C_preamp100_feb26.csv
 2 : calibration/daily/Co60_22.5C_preamp100_mar3.csv
 3 : calibration/daily/Cs_21.9C_preamp0_mar5.csv
 4 : calibration/daily/Cs_22.0C_preamp0_mar3.csv
 5 : calibration/daily/Cs_22.4C_preamp0_feb26.csv
 6 : calibration/feb_24/preamp_0/Ba_22.4C_preamp0_feb24.csv
 7 : calibration/feb_24/preamp_0/Cs_22.1C_preamp0_feb24.csv
 8 : calibration/feb_24/preamp_100/Co60_22.5C_preamp100_feb24.csv
 9 : calibration/feb_24/preamp_100/Na_22.5C_preamp100_feb24.csv
10 : data/Co57_runs_overnight/Co57_22.2C_preamp0_feb25_overnight.csv
11 : data/feb_24/Na22/Na_22.5C_feb24_run1.csv
12 : data/feb_24/co60/Co_22.4C_feb24_run1.csv
13 : data/feb_26/co60/Co_22.8C_feb26_run1.csv
14 : data/feb_26/cs137/Cs_22.4C_feb26_run1.csv
15 : data/feb_26/na22/Na_22.6C_feb26_run1.csv
16 : data/mar_3/co60/Co_22.5C_mar3_run1.csv
17 : data/mar_3/cs137/Cs_22.5C_mar3_run1.csv
18 : data/mar_3/na22/Na

In [7]:
# -----------------------------
# INTERACTIVE FIT GROUP SELECTOR
# -----------------------------

SELECTION_STATE = {
    "x": None,
    "y": None,
    "title": None,
    "fig": None,
    "ax": None,
    "cid_click": None,
    "cid_key": None,
    "temp_clicks": [],
    "groups": [],
    "temp_artists": [],
    "group_artists": [],
    "status_text": None,
}


def nearest_index(xarr, xclick):
    return int(np.argmin(np.abs(xarr - xclick)))


def _clear_temp_artists():
    for artist in SELECTION_STATE["temp_artists"]:
        try:
            artist.remove()
        except Exception:
            pass
    SELECTION_STATE["temp_artists"] = []


def _update_status_text():
    ax = SELECTION_STATE["ax"]
    if ax is None:
        return

    temp_n = len(SELECTION_STATE["temp_clicks"])
    group_n = len(SELECTION_STATE["groups"])

    message = (
        f"Stored groups: {group_n}\n"
        f"Current clicks: {temp_n}\n"
        "Controls: left click add | n store group | u undo | d delete last group | r reset current | q finish"
    )

    if SELECTION_STATE["status_text"] is None:
        SELECTION_STATE["status_text"] = ax.text(
            0.02, 0.98, message,
            transform=ax.transAxes,
            va="top",
            bbox=dict(facecolor="white", alpha=0.85, edgecolor="gray")
        )
    else:
        SELECTION_STATE["status_text"].set_text(message)

    SELECTION_STATE["fig"].canvas.draw_idle()


def _draw_temp_clicks():
    ax = SELECTION_STATE["ax"]
    x = SELECTION_STATE["x"]
    y = SELECTION_STATE["y"]

    _clear_temp_artists()

    for j, (_, _, idx) in enumerate(SELECTION_STATE["temp_clicks"], start=1):
        xs = float(x[idx])
        ys = float(y[idx])

        line = ax.axvline(xs, linestyle=":", alpha=0.8)
        marker = ax.plot(xs, ys, marker="o", ms=7)[0]
        text = ax.text(xs, ys, f"T{j}", fontsize=9, va="bottom", ha="left")

        SELECTION_STATE["temp_artists"].extend([line, marker, text])

    _update_status_text()


def _draw_stored_group(group):
    ax = SELECTION_STATE["ax"]
    x = SELECTION_STATE["x"]
    y = SELECTION_STATE["y"]

    artists = []

    xmin = group["xmin"]
    xmax = group["xmax"]
    centers = group["centers"]
    group_id = group["group_id"]

    span = ax.axvspan(xmin, xmax, alpha=0.15)
    artists.append(span)

    left_line = ax.axvline(xmin, linestyle="--", alpha=0.9)
    right_line = ax.axvline(xmax, linestyle="--", alpha=0.9)
    artists.extend([left_line, right_line])

    mask = (x >= xmin) & (x <= xmax)
    ymid = np.nanmax(y[mask]) if np.any(mask) else np.nanmax(y)
    label = ax.text((xmin + xmax) / 2, ymid, f"G{group_id}", ha="center", va="bottom")
    artists.append(label)

    for k, center in enumerate(centers, start=1):
        idx = nearest_index(x, center)
        xs = float(x[idx])
        ys = float(y[idx])

        center_line = ax.axvline(xs, linestyle="-.", alpha=0.9)
        center_marker = ax.plot(xs, ys, marker="o", ms=8)[0]
        center_text = ax.text(xs, ys, f"G{group_id}P{k}", fontsize=9, va="bottom", ha="left")

        artists.extend([center_line, center_marker, center_text])

    SELECTION_STATE["group_artists"].append(artists)
    _update_status_text()


def _store_current_group():
    temp = SELECTION_STATE["temp_clicks"]

    if len(temp) < 3:
        print("Need at least 3 clicks to store a group: left boundary, one or more centers, right boundary.")
        return

    xs = sorted([float(row[0]) for row in temp])

    xmin = xs[0]
    xmax = xs[-1]
    centers = xs[1:-1]

    if len(centers) == 0:
        print("A group must contain at least one peak center.")
        return

    group = {
        "group_id": len(SELECTION_STATE["groups"]) + 1,
        "xmin": float(xmin),
        "xmax": float(xmax),
        "centers": [float(c) for c in centers],
        "n_peaks": len(centers),
    }

    SELECTION_STATE["groups"].append(group)
    print(f"Stored group {group['group_id']}: xmin={xmin:.3f}, centers={centers}, xmax={xmax:.3f}")

    SELECTION_STATE["temp_clicks"] = []
    _clear_temp_artists()
    _draw_stored_group(group)


def _delete_last_group():
    if len(SELECTION_STATE["groups"]) == 0:
        print("No stored groups to delete.")
        return

    SELECTION_STATE["groups"].pop()

    artists = SELECTION_STATE["group_artists"].pop()
    for artist in artists:
        try:
            artist.remove()
        except Exception:
            pass

    for i, group in enumerate(SELECTION_STATE["groups"], start=1):
        group["group_id"] = i

    print("Deleted last stored group.")
    _update_status_text()


def _reset_current_group():
    SELECTION_STATE["temp_clicks"] = []
    _clear_temp_artists()
    print("Cleared current unfinished group.")
    _update_status_text()


def _undo_last_temp_click():
    if len(SELECTION_STATE["temp_clicks"]) == 0:
        print("No current clicks to undo.")
        return

    removed = SELECTION_STATE["temp_clicks"].pop()
    print(f"Removed last temporary click at x={removed[0]:.3f}")
    _draw_temp_clicks()


def _on_click(event):
    ax = SELECTION_STATE["ax"]
    x = SELECTION_STATE["x"]
    y = SELECTION_STATE["y"]

    if event.inaxes != ax or event.xdata is None:
        return

    idx = nearest_index(x, event.xdata)
    xs = float(x[idx])
    ys = float(y[idx])

    SELECTION_STATE["temp_clicks"].append((xs, ys, idx))
    print(f"Added temporary click #{len(SELECTION_STATE['temp_clicks'])}: x={xs:.6f}, y={ys:.6f}, idx={idx}")
    _draw_temp_clicks()


def _on_key(event):
    if event.key == "n":
        _store_current_group()
    elif event.key == "u":
        _undo_last_temp_click()
    elif event.key == "d":
        _delete_last_group()
    elif event.key == "r":
        _reset_current_group()
    elif event.key == "h":
        print("Controls:")
        print("  left click -> add point")
        print("  n -> store current group")
        print("  u -> undo last temp click")
        print("  d -> delete last stored group")
        print("  r -> reset current unfinished group")
        print("  q -> finish and close window")
    elif event.key == "q":
        print("Finishing selection and closing window.")
        plt.close(SELECTION_STATE["fig"])


def launch_group_selector(x, y, title="Select fit groups"):
    """
    Each group is defined by temporary clicks.
    For one group, click:
        left boundary, one or more peak centers, right boundary
    in any order, then press 'n' to store the group.
    """

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    SELECTION_STATE["x"] = x
    SELECTION_STATE["y"] = y
    SELECTION_STATE["title"] = title
    SELECTION_STATE["temp_clicks"] = []
    SELECTION_STATE["groups"] = []
    SELECTION_STATE["temp_artists"] = []
    SELECTION_STATE["group_artists"] = []
    SELECTION_STATE["status_text"] = None

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(x, y, lw=1.2)
    ax.set_xlabel("ADC Channel")
    ax.set_ylabel("Background-subtracted rate (counts/s)")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

    positive_y = y[y > 0]
    if len(positive_y) > 0:
        ymin = max(np.nanmin(positive_y), 1e-6)
        ymax = np.nanmax(positive_y)
        ax.set_yscale("log")
        ax.set_ylim(ymin, ymax * 1.25 if ymax > ymin else ymin * 10)
    else:
        ymin = np.nanmin(y)
        ymax = np.nanmax(y)
        yrange = ymax - ymin if ymax > ymin else 1.0
        ax.set_ylim(ymin - 0.05 * yrange, ymax + 0.12 * yrange)

    SELECTION_STATE["fig"] = fig
    SELECTION_STATE["ax"] = ax
    SELECTION_STATE["cid_click"] = fig.canvas.mpl_connect("button_press_event", _on_click)
    SELECTION_STATE["cid_key"] = fig.canvas.mpl_connect("key_press_event", _on_key)

    _update_status_text()

    print("A separate plot window should open.")
    print("The selector plot uses a log-linear y-axis when positive data are available.")
    print("Define one fit group at a time.")
    print("For each group, click: left boundary, one or more peak centers, right boundary.")
    print("Then press 'n' to store that group.")
    print("Press 'q' when all groups are stored.")

    plt.show()


def get_fit_groups():
    groups = []
    for group in SELECTION_STATE["groups"]:
        groups.append({
            "group_id": int(group["group_id"]),
            "xmin": float(group["xmin"]),
            "xmax": float(group["xmax"]),
            "centers": [float(c) for c in group["centers"]],
            "n_peaks": int(group["n_peaks"]),
        })
    return groups


def clear_selection_state():
    SELECTION_STATE["temp_clicks"] = []
    SELECTION_STATE["groups"] = []
    _clear_temp_artists()
    print("Selection state cleared.")

In [8]:
# -----------------------------
# FITTING ROUTINES
# -----------------------------

def fit_group_multi_gaussian(x, y, group, spectrum_label="current spectrum"):
    xmin = float(group["xmin"])
    xmax = float(group["xmax"])
    centers = [float(c) for c in group["centers"]]
    group_id = int(group["group_id"])

    mask = (x >= xmin) & (x <= xmax)
    xfit = np.asarray(x[mask], dtype=float)
    yfit = np.asarray(y[mask], dtype=float)

    if len(xfit) < 5:
        raise RuntimeError(f"Group {group_id}: not enough points inside fit window.")

    n_peaks = len(centers)
    window = float(xmax - xmin)

    left_edge_n = max(2, len(yfit) // 10)
    right_edge_n = max(2, len(yfit) // 10)

    x_left = np.median(xfit[:left_edge_n])
    x_right = np.median(xfit[-right_edge_n:])
    y_left = np.median(yfit[:left_edge_n])
    y_right = np.median(yfit[-right_edge_n:])

    if x_right != x_left:
        m0 = (y_right - y_left) / (x_right - x_left)
    else:
        m0 = 0.0

    b0 = float(np.median(yfit) - m0 * np.median(xfit))

    p0 = []
    lower = []
    upper = []

    baseline_guess = m0 * xfit + b0

    for center in centers:
        idx_center = nearest_index(xfit, center)
        A0 = float(yfit[idx_center] - baseline_guess[idx_center])
        if not np.isfinite(A0) or A0 <= 0:
            A0 = float(np.max(yfit) - np.min(yfit))
        if A0 <= 0:
            A0 = 1.0

        sigma0 = max(window / (8.0 * max(n_peaks, 1)), 1.0)

        p0.extend([A0, center, sigma0])
        lower.extend([0.0, xmin, 0.5])
        upper.extend([np.inf, xmax, max(window, 2.0)])

    p0.extend([m0, b0])
    lower.extend([-np.inf, -np.inf])
    upper.extend([np.inf, np.inf])

    popt, pcov = curve_fit(
        multi_gaussian_with_background,
        xfit,
        yfit,
        p0=p0,
        bounds=(lower, upper),
        maxfev=50000,
    )

    perr = np.sqrt(np.diag(pcov)) if pcov is not None else np.full(len(popt), np.nan)

    xs_dense = np.linspace(xmin, xmax, 600)
    y_total = multi_gaussian_with_background(xs_dense, *popt)

    component_rows = []
    component_curves = []

    for i in range(n_peaks):
        A = float(popt[3 * i + 0])
        mu = float(popt[3 * i + 1])
        sigma = float(abs(popt[3 * i + 2]))

        A_err = float(perr[3 * i + 0]) if len(perr) > 3 * i + 0 else np.nan
        mu_err = float(perr[3 * i + 1]) if len(perr) > 3 * i + 1 else np.nan
        sigma_err = float(perr[3 * i + 2]) if len(perr) > 3 * i + 2 else np.nan

        y_component = gaussian(xs_dense, A, mu, sigma)
        component_curves.append((i + 1, xs_dense, y_component))

        component_rows.append({
            "file": spectrum_label,
            "group_id": group_id,
            "peak_in_group": i + 1,
            "n_peaks_in_group": n_peaks,
            "clicked_center_ADC": float(centers[i]),
            "center_ADC": mu,
            "center_ADC_err": mu_err,
            "sigma_ADC": sigma,
            "sigma_ADC_err": sigma_err,
            "amplitude_counts_per_s": A,
            "amplitude_counts_per_s_err": A_err,
            "fit_xmin": xmin,
            "fit_xmax": xmax,
            "background_slope": float(popt[-2]),
            "background_intercept": float(popt[-1]),
            "gaussian_area_counts": float(A * sigma * np.sqrt(2.0 * np.pi)),
            "theoretical_energy_keV": np.nan,
            "theoretical_energy_keV_err": np.nan,
        })

    fit_summary = {
        "group_id": group_id,
        "xmin": xmin,
        "xmax": xmax,
        "xfit": xfit,
        "yfit": yfit,
        "xs_dense": xs_dense,
        "y_total": y_total,
        "component_curves": component_curves,
        "rows": component_rows,
        "popt": popt,
        "pcov": pcov,
    }

    return fit_summary


def fit_all_groups(x, y, fit_groups, spectrum_label="current spectrum"):
    fit_summaries = []
    result_rows = []

    for group in fit_groups:
        try:
            summary = fit_group_multi_gaussian(x, y, group, spectrum_label=spectrum_label)
            fit_summaries.append(summary)
            result_rows.extend(summary["rows"])
            print(f"Fit succeeded for group {group['group_id']} with {group['n_peaks']} peak(s).")
        except Exception as e:
            print(f"Fit failed for group {group['group_id']}: {e}")

    return fit_summaries, result_rows

In [9]:
# -----------------------------
# PLOT HELPERS
# -----------------------------

def _positive_ylim_from_arrays(*arrays, floor=1e-6, expansion_top=1.25):
    vals = []
    for arr in arrays:
        arr = np.asarray(arr, dtype=float)
        vals.extend(arr[np.isfinite(arr) & (arr > 0)])
    vals = np.asarray(vals, dtype=float)

    if len(vals) == 0:
        return None

    ymin = max(np.min(vals), floor)
    ymax = np.max(vals)

    if ymax <= ymin:
        ymax = ymin * 10.0

    return ymin, ymax * expansion_top


def plot_documentation_figure(x, raw_rate, background_rate, adjusted_rate, title):
    fig, ax = plt.subplots(figsize=(10, 6))

    ax.plot(x, raw_rate, label="Raw sample rate")
    ax.plot(x, background_rate, label="Background rate")
    ax.plot(x, adjusted_rate, label="Background-subtracted rate")

    ax.set_xlabel("ADC Channel")
    ax.set_ylabel("Rate (counts/s)")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

    pos_ylim = _positive_ylim_from_arrays(raw_rate, background_rate, adjusted_rate)
    if pos_ylim is not None:
        ax.set_yscale("log")
        ax.set_ylim(*pos_ylim)
        ax.legend(title="log-linear")
    else:
        ax.legend(title="linear (fallback)")

    plt.show()
    return fig, ax


def plot_fit_results(x, adjusted_rate, fit_summaries, spectrum_label="current spectrum"):
    fig, ax = plt.subplots(figsize=(12, 6))

    ax.plot(x, adjusted_rate, label="Background-subtracted spectrum", lw=1.2)

    all_positive_components = [adjusted_rate]

    for summary in fit_summaries:
        group_id = summary["group_id"]
        xmin = summary["xmin"]
        xmax = summary["xmax"]

        ax.axvspan(xmin, xmax, alpha=0.12)
        ax.plot(summary["xs_dense"], summary["y_total"], linewidth=2, label=f"Group {group_id} total fit")
        all_positive_components.append(summary["y_total"])

        for peak_in_group, xs_dense, y_component in summary["component_curves"]:
            ax.plot(xs_dense, y_component, linestyle="--", linewidth=1.5, label=f"G{group_id} peak {peak_in_group}")
            all_positive_components.append(y_component)

    ax.set_xlabel("ADC Channel")
    ax.set_ylabel("Background-subtracted rate (counts/s)")
    ax.set_title(f"{spectrum_label}\nBackground-subtracted spectrum with fitted Gaussian group(s)")
    ax.grid(True, alpha=0.3)

    pos_ylim = _positive_ylim_from_arrays(*all_positive_components)
    if pos_ylim is not None:
        ax.set_yscale("log")
        ax.set_ylim(*pos_ylim)
        ax.legend(title="log-linear")
    else:
        ax.legend(title="linear (fallback)")

    plt.show()
    return fig, ax

In [563]:
# -----------------------------
# START: CHOOSE CURRENT FILE
# -----------------------------

file_index = int(input("Enter file index from the list above: "))

current_file = candidate_files[file_index]
current_rel = normalize_relpath(current_file)

(
    x_current,
    raw_rate_current,
    background_rate_current,
    adjusted_rate_current,
    sample_meta_current,
    background_meta_current,
) = compute_background_subtracted_rate(current_file, background_path=BACKGROUND_FILE)

print("Current file:", current_rel)
print("Sample elapsed time (s):", sample_meta_current.get("Elapsed Real Time (s)"))
print("Background elapsed time (s):", background_meta_current.get("Elapsed Real Time (s)"))

Enter file index from the list above:  37


Current file: data/thickness_effects_in_front/Pb_11cmpmt_tosource_5cmthick.csv
Sample elapsed time (s): 912.0
Background elapsed time (s): 418963.0


In [564]:
# -----------------------------
# DOCUMENTATION PLOT
# -----------------------------

fig_doc, ax_doc = plot_documentation_figure(
    x_current,
    raw_rate_current,
    background_rate_current,
    adjusted_rate_current,
    title=f"{current_rel}\nRaw vs background vs background-subtracted"
)

In [565]:
# -----------------------------
# RUN FIT GROUP SELECTOR
# -----------------------------

launch_group_selector(
    x_current,
    adjusted_rate_current,
    title=f"{current_rel} | background-subtracted only | define fit groups"
)

A separate plot window should open.
The selector plot uses a log-linear y-axis when positive data are available.
Define one fit group at a time.
For each group, click: left boundary, one or more peak centers, right boundary.
Then press 'n' to store that group.
Press 'q' when all groups are stored.
Added temporary click #1: x=438.000000, y=0.085695, idx=438
Added temporary click #2: x=502.000000, y=1.585493, idx=502
Added temporary click #3: x=566.000000, y=0.016450, idx=566
Stored group 1: xmin=438.000, centers=[502.0], xmax=566.000
Deleted last stored group.
Added temporary click #1: x=447.000000, y=0.074037, idx=447
Added temporary click #2: x=503.000000, y=1.598093, idx=503
Added temporary click #3: x=560.000000, y=0.015481, idx=560
Stored group 1: xmin=447.000, centers=[503.0], xmax=560.000
Finishing selection and closing window.


In [566]:
# -----------------------------
# INSPECT FIT GROUPS
# -----------------------------

fit_groups_current = get_fit_groups()

print("fit_groups_current:")
for group in fit_groups_current:
    print(group)

print("Number of stored groups:", len(fit_groups_current))

fit_groups_current:
{'group_id': 1, 'xmin': 447.0, 'xmax': 560.0, 'centers': [503.0], 'n_peaks': 1}
Number of stored groups: 1


In [567]:
# -----------------------------
# OPTIONAL TYPED FALLBACK
# -----------------------------
# Only run this if you want to overwrite fit_groups_current manually.

def manual_group_entry():
    groups = []

    n_groups = int(input("Number of fit groups: "))

    for i in range(n_groups):
        print(f"\nGroup {i+1}")
        xmin = float(input("xmin: "))
        xmax = float(input("xmax: "))

        if xmax < xmin:
            xmin, xmax = xmax, xmin

        n_peaks = int(input("Number of peaks in this group: "))
        centers = []

        for j in range(n_peaks):
            c = float(input(f"Center guess for peak {j+1}: "))
            centers.append(c)

        groups.append({
            "group_id": i + 1,
            "xmin": float(xmin),
            "xmax": float(xmax),
            "centers": [float(c) for c in centers],
            "n_peaks": int(n_peaks),
        })

    return groups

# Uncomment if needed:
# fit_groups_current = manual_group_entry()

In [568]:
# -----------------------------
# SANITY CHECK
# -----------------------------

if "fit_groups_current" not in globals():
    raise RuntimeError("fit_groups_current does not exist. Run the selector cell first.")

if len(fit_groups_current) == 0:
    raise RuntimeError("No fit groups were stored. Use the selector and press 'n' for at least one group.")

print("Ready to fit.")

Ready to fit.


In [569]:
# -----------------------------
# FIT ALL GROUPS
# -----------------------------

fit_summaries_current, fit_rows_current = fit_all_groups(
    x_current,
    adjusted_rate_current,
    fit_groups_current,
    spectrum_label=current_rel
)

print("\nFit rows:")
for row in fit_rows_current:
    print(row)

Fit succeeded for group 1 with 1 peak(s).

Fit rows:
{'file': 'data/thickness_effects_in_front/Pb_11cmpmt_tosource_5cmthick.csv', 'group_id': 1, 'peak_in_group': 1, 'n_peaks_in_group': 1, 'clicked_center_ADC': 503.0, 'center_ADC': 507.86987123576085, 'center_ADC_err': 0.24390228659087387, 'sigma_ADC': 14.337767553459038, 'sigma_ADC_err': 0.28945458393445833, 'amplitude_counts_per_s': 1.7092374077670514, 'amplitude_counts_per_s_err': 0.025474191867768003, 'fit_xmin': 447.0, 'fit_xmax': 560.0, 'background_slope': -0.0012461789923829665, 'background_intercept': 0.680546203085073, 'gaussian_area_counts': 61.42905841311488, 'theoretical_energy_keV': nan, 'theoretical_energy_keV_err': nan}


In [570]:
# -----------------------------
# ASSIGN REFERENCE ENERGIES (keV) + ERRORS
# -----------------------------

def assign_theoretical_energies(result_rows):
    """
    Assign reference gamma energies and optional energy uncertainties by hand.

    This is done row by row in the current fitted order:
      group_id, then peak_in_group

    Notes:
    - leave energy blank to skip the peak entirely
    - leave energy error blank if no reported uncertainty is available
    """

    if len(result_rows) == 0:
        print("No fitted peaks available to annotate.")
        return result_rows

    print("Assign reference energies peak-by-peak.")
    print("Leave energy blank to skip a peak.")
    print("Leave energy error blank if no reported uncertainty is available.")

    for i, row in enumerate(result_rows, start=1):
        print("\n----------------------------------------")
        print(f"Row {i}")
        print("file:", row["file"])
        print("group_id:", row["group_id"])
        print("peak_in_group:", row["peak_in_group"])
        print(f"fitted center_ADC: {row['center_ADC']:.6f}")
        print(f"clicked_center_ADC: {row['clicked_center_ADC']:.6f}")

        energy_text = input("Reference energy keV (blank to skip): ").strip()
        energy_err_text = input("Reference energy error keV (blank if none reported): ").strip()

        if energy_text == "":
            row["theoretical_energy_keV"] = np.nan
            row["theoretical_energy_keV_err"] = np.nan
        else:
            row["theoretical_energy_keV"] = float(energy_text)

            if energy_err_text == "":
                row["theoretical_energy_keV_err"] = np.nan
            else:
                row["theoretical_energy_keV_err"] = float(energy_err_text)

    return result_rows


fit_rows_current = assign_theoretical_energies(fit_rows_current)

Assign reference energies peak-by-peak.
Leave energy blank to skip a peak.
Leave energy error blank if no reported uncertainty is available.

----------------------------------------
Row 1
file: data/thickness_effects_in_front/Pb_11cmpmt_tosource_5cmthick.csv
group_id: 1
peak_in_group: 1
fitted center_ADC: 507.869871
clicked_center_ADC: 503.000000


Reference energy keV (blank to skip):  661.657
Reference energy error keV (blank if none reported):  0.003


In [571]:
# -----------------------------
# PLOT FITTED RESULT
# -----------------------------

fig_fit, ax_fit = plot_fit_results(
    x_current,
    adjusted_rate_current,
    fit_summaries_current,
    spectrum_label=current_rel
)

In [572]:
# -----------------------------
# SAVE FITTED PLOT
# -----------------------------

FIT_PLOT_DIR.mkdir(parents=True, exist_ok=True)

fit_plot_file = FIT_PLOT_DIR / f"{Path(current_rel).stem}_fit_groups.png"
fig_fit.savefig(fit_plot_file, dpi=200, bbox_inches="tight")

print("Saved fitted plot to:", fit_plot_file)

Saved fitted plot to: peak_analysis_outputs/fit_plots/Pb_11cmpmt_tosource_5cmthick_fit_groups.png


In [573]:
# -----------------------------
# BUILD RESULTS DATAFRAME
# -----------------------------

results_df = pd.DataFrame(fit_rows_current)

preferred_cols = [
    "file",
    "group_id",
    "peak_in_group",
    "n_peaks_in_group",
    "theoretical_energy_keV",
    "theoretical_energy_keV_err",
    "clicked_center_ADC",
    "center_ADC",
    "center_ADC_err",
    "sigma_ADC",
    "sigma_ADC_err",
    "amplitude_counts_per_s",
    "amplitude_counts_per_s_err",
    "gaussian_area_counts",
    "fit_xmin",
    "fit_xmax",
    "background_slope",
    "background_intercept",
]

existing_cols = [c for c in preferred_cols if c in results_df.columns]
results_df = results_df[existing_cols]

print(results_df)
print("\nNumber of fitted peaks:", len(results_df))

                                                file  group_id  peak_in_group  \
0  data/thickness_effects_in_front/Pb_11cmpmt_tos...         1              1   

   n_peaks_in_group  theoretical_energy_keV  theoretical_energy_keV_err  \
0                 1                 661.657                       0.003   

   clicked_center_ADC  center_ADC  center_ADC_err  sigma_ADC  sigma_ADC_err  \
0               503.0  507.869871        0.243902  14.337768       0.289455   

   amplitude_counts_per_s  amplitude_counts_per_s_err  gaussian_area_counts  \
0                1.709237                    0.025474             61.429058   

   fit_xmin  fit_xmax  background_slope  background_intercept  
0     447.0     560.0         -0.001246              0.680546  

Number of fitted peaks: 1


In [574]:
# -----------------------------
# SAVE PER-RUN RESULTS
# -----------------------------

PER_RUN_DIR.mkdir(parents=True, exist_ok=True)

out_file = PER_RUN_DIR / f"{Path(current_rel).stem}_manual_peak_fits.csv"
results_df.to_csv(out_file, index=False)

print("Saved peak fits to:", out_file)

Saved peak fits to: peak_analysis_outputs/per_run_csv/Pb_11cmpmt_tosource_5cmthick_manual_peak_fits.csv


In [575]:
# -----------------------------
# APPEND TO MASTER FILE
# -----------------------------

if MASTER_FILE.exists():
    old_df = pd.read_csv(MASTER_FILE)

    # replace old rows for this file rather than blindly duplicating them
    old_df = old_df[old_df["file"] != current_rel]
    combined_df = pd.concat([old_df, results_df], ignore_index=True)
else:
    combined_df = results_df.copy()

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
combined_df.to_csv(MASTER_FILE, index=False)

print("Updated master file:", MASTER_FILE)
print("Rows in master file:", len(combined_df))

Updated master file: peak_analysis_outputs/manual_peak_fits_all.csv
Rows in master file: 138


In [586]:
# -----------------------------
# CLEAR INTERACTIVE STATE
# -----------------------------

clear_selection_state()
print("Interactive state cleared.")

Selection state cleared.
Interactive state cleared.


In [24]:
# -----------------------------
# FULL CLEANUP / RESET
# -----------------------------

def clear_all_peak_analysis_outputs(
    remove_output_directory=True,
    clear_interactive_state=True,
    clear_runtime_globals=False,
):
    """
    Use this when you want to restart peak analysis cleanly.

    Options:
    - remove_output_directory:
        delete peak_analysis_outputs entirely
    - clear_interactive_state:
        clear current click/group selector memory
    - clear_runtime_globals:
        delete common current-run variables from globals()
    """

    if clear_interactive_state:
        try:
            clear_selection_state()
        except Exception as e:
            print("Could not clear selection state:", e)

    if remove_output_directory:
        if OUTPUT_ROOT.exists():
            shutil.rmtree(OUTPUT_ROOT)
            print("Deleted output directory:", OUTPUT_ROOT)
        else:
            print("Output directory does not exist:", OUTPUT_ROOT)

    if clear_runtime_globals:
        names_to_clear = [
            "current_file",
            "current_rel",
            "x_current",
            "raw_rate_current",
            "background_rate_current",
            "adjusted_rate_current",
            "sample_meta_current",
            "background_meta_current",
            "fit_groups_current",
            "fit_summaries_current",
            "fit_rows_current",
            "results_df",
            "fig_doc",
            "ax_doc",
            "fig_fit",
            "ax_fit",
            "doc_plot_file",
            "fit_plot_file",
            "out_file",
        ]

        for name in names_to_clear:
            if name in globals():
                del globals()[name]

        print("Cleared common runtime variables from globals().")

    print("Peak analysis reset complete.")

In [25]:
# -----------------------------
# RESET CALL
# -----------------------------
# Uncomment to fully wipe prior outputs and current in-memory state.
# Run this ONLY when you really want to restart cleanly.

#clear_all_peak_analysis_outputs(
#     remove_output_directory=True,
#     clear_interactive_state=True,
#     clear_runtime_globals=True,
#)